# Siri Portfolio Chatbot — Semantic RAG Pipeline

**Architecture:** `about_me.json → Sentence Transformers → ChromaDB → Reranker → Groq → Response`

This notebook contains the complete pipeline developed so far, with markdown explanations and clean, separated cells.

> Keep `about_me.json` in the same folder as this notebook, or change `JSON_PATH`.


## 1. Install dependencies

Run once if needed:

```bash
pip install chromadb sentence-transformers groq scikit-learn
```


In [5]:
# Uncomment and run once if needed:
# !pip install chromadb sentence-transformers groq scikit-learn

## 2. Imports and configuration

Separate names are used for the embedding model and Groq client to prevent variable-name collisions.


In [6]:
import os
import json
import chromadb
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

JSON_PATH = "data/about_me.json"
CHROMA_PATH = "data/chroma_db"
COLLECTION_NAME = "siri_knowledge"

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
GROQ_MODEL_NAME = "openai/gpt-oss-20b"

print("Imports loaded.")


c:\Users\shres\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded.


## 3. Initialize models and clients

Set `GROQ_API_KEY` in your environment before running this cell.

If you already have a Groq client, you can replace this initialization with your existing client.


In [7]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
groq_client = Groq(api_key=[GROQ_API_KEY])

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Groq model: {GROQ_MODEL_NAME}")


Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


Embedding model: all-MiniLM-L6-v2
Groq model: openai/gpt-oss-20b


## 4. Load and flatten the knowledge base

The JSON can be organized as a dictionary of lists rather than one flat list, so this helper safely produces one list of chunks.


In [8]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Raw JSON type:", type(data).__name__)


Raw JSON type: dict


In [9]:
def get_all_chunks(data):
    if isinstance(data, list):
        return [
            item for item in data
            if isinstance(item, dict) and "id" in item
        ]

    chunks = []

    if isinstance(data, dict):
        for value in data.values():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and "id" in item:
                        chunks.append(item)
            elif isinstance(value, dict) and "id" in value:
                chunks.append(value)

    return chunks


all_chunks = get_all_chunks(data)

print(f"Loaded {len(all_chunks)} knowledge chunks.")


Loaded 97 knowledge chunks.


## 5. Build embedding documents

Each document contains the topic, question patterns, keywords, and answer. Question patterns help semantic retrieval handle natural variations in wording.


In [10]:
documents = []

for item in all_chunks:
    text = f"""
Topic: {item.get('topic', '')}

Questions:
{chr(10).join(item.get('question_patterns', []))}

Keywords:
{', '.join(item.get('keywords', []))}

Answer:
{item.get('answer', '')}
""".strip()

    documents.append(text)

print(f"Created {len(documents)} embedding documents.")


Created 97 embedding documents.


## 6. Generate embeddings

`all-MiniLM-L6-v2` creates compact semantic embeddings for local retrieval.


In [11]:
embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)


Batches: 100%|██████████| 4/4 [00:01<00:00,  2.57it/s]

Embedding shape: (97, 384)


## 7. Rebuild the ChromaDB collection

Run this section whenever `about_me.json` changes.

This deletes only the derived Chroma collection; it does not modify the JSON knowledge base.


In [12]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("Old collection deleted.")
except Exception:
    print("No existing collection found.")

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME
)

print(f"Collection ready: {COLLECTION_NAME}")


Old collection deleted.
Collection ready: siri_knowledge


In [13]:
ids = [item["id"] for item in all_chunks]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

print(f"Inserted {len(ids)} chunks into ChromaDB.")


Inserted 97 chunks into ChromaDB.


## 8. Basic semantic retrieval

Chroma retrieves several candidates. We then rerank those candidates instead of trusting Chroma's top result blindly.


In [14]:
def semantic_search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    return collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )


def print_semantic_results(query, top_k=5):
    results = semantic_search(query, top_k)

    print(f"Query: {query}\n")

    for i, (chunk_id, distance) in enumerate(
        zip(results["ids"][0], results["distances"][0]),
        1
    ):
        print(f"{i}. {chunk_id} | distance: {distance:.4f}")


## 9. Reranker

The final score combines:

- 55% semantic similarity
- 30% question-pattern similarity
- 15% keyword overlap

This fixes cases where the correct chunk is already among Chroma's top candidates but is not ranked first.


In [15]:
def rerank(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    candidate_ids = results["ids"][0]
    distances = results["distances"][0]

    data_map = {
        item["id"]: item
        for item in all_chunks
    }

    candidates = []

    for i, chunk_id in enumerate(candidate_ids):
        item = data_map.get(chunk_id)

        if item is None:
            continue

        pattern_text = " ".join(
            item.get("question_patterns", [])
        )

        keyword_text = " ".join(
            item.get("keywords", [])
        )

        if pattern_text.strip():
            vectorizer = TfidfVectorizer(
                ngram_range=(1, 2),
                lowercase=True
            )

            vectors = vectorizer.fit_transform([
                query,
                pattern_text
            ])

            pattern_score = cosine_similarity(
                vectors[0:1],
                vectors[1:2]
            )[0][0]
        else:
            pattern_score = 0.0

        query_words = set(query.lower().split())
        keywords = set(keyword_text.lower().split())

        keyword_score = (
            len(query_words & keywords)
            / max(len(query_words), 1)
        )

        chroma_distance = distances[i]
        semantic_score = 1 / (1 + chroma_distance)

        final_score = (
            0.55 * semantic_score +
            0.30 * pattern_score +
            0.15 * keyword_score
        )

        candidates.append({
            "id": chunk_id,
            "score": final_score,
            "semantic": semantic_score,
            "pattern": pattern_score,
            "keyword": keyword_score
        })

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return candidates


In [16]:
def print_rerank_results(query, top_k=5):
    results = rerank(query, top_k)

    print(f"Query: {query}\n")

    for i, result in enumerate(results, 1):
        print(
            f"{i}. {result['id']} | "
            f"score={result['score']:.3f} | "
            f"semantic={result['semantic']:.3f} | "
            f"pattern={result['pattern']:.3f} | "
            f"keyword={result['keyword']:.3f}"
        )


## 10. Retrieval test suite

The expected IDs below are the targets established during development.


In [17]:
test_cases = [
    ("Who am I speaking with?", "identity_001"),
    ("Tell me about yourself.", "story_one_paragraph_001"),
    ("What are you currently studying?", "education_bca_001"),
    ("Why did you move from medicine toward computers?", "education_career_transition_001"),
    ("Tell me about the biometric thing you built for Windows.", "project_faceunlock_001"),
    ("How does your face authentication system recognize someone?", "project_faceunlock_001"),
    ("What is your AI assistant?", "project_tars_001"),
    ("What technologies can you program in?", "skills_programming_languages_001"),
    ("What projects have you built?", "projects_overview_001"),
    ("What programming languages do you know?", "skills_programming_languages_001"),
    ("What are your career goals?", "career_direction_001"),
    ("What did you build with ESP32?", "skills_hardware_001"),
]

correct = 0

for question, expected_id in test_cases:
    results = rerank(question, top_k=5)
    predicted_id = results[0]["id"] if results else None
    is_correct = predicted_id == expected_id
    correct += int(is_correct)

    print(f"{'✓' if is_correct else '✗'} Q: {question}")
    print(f"  Expected: {expected_id}")
    print(f"  Got:      {predicted_id}\n")

print(
    f"Accuracy: {correct}/{len(test_cases)} "
    f"({correct / len(test_cases) * 100:.1f}%)"
)


✓ Q: Who am I speaking with?
  Expected: identity_001
  Got:      identity_001

✓ Q: Tell me about yourself.
  Expected: story_one_paragraph_001
  Got:      story_one_paragraph_001

✓ Q: What are you currently studying?
  Expected: education_bca_001
  Got:      education_bca_001

✓ Q: Why did you move from medicine toward computers?
  Expected: education_career_transition_001
  Got:      education_career_transition_001

✓ Q: Tell me about the biometric thing you built for Windows.
  Expected: project_faceunlock_001
  Got:      project_faceunlock_001

✓ Q: How does your face authentication system recognize someone?
  Expected: project_faceunlock_001
  Got:      project_faceunlock_001

✓ Q: What is your AI assistant?
  Expected: project_tars_001
  Got:      project_tars_001

✓ Q: What technologies can you program in?
  Expected: skills_programming_languages_001
  Got:      skills_programming_languages_001

✓ Q: What projects have you built?
  Expected: projects_overview_001
  Got:      p

## 11. Groq response generation

The LLM does not retrieve facts. The reranker selects the best knowledge chunk first; Groq only turns that grounded answer into a natural first-person response.


In [18]:
def generate_response(query):
    results = rerank(query, top_k=5)

    if not results:
        return "I don't have enough information to answer that."

    best_id = results[0]["id"]

    chunk = next(
        (
            item for item in all_chunks
            if item["id"] == best_id
        ),
        None
    )

    if chunk is None:
        return "I don't have enough information to answer that."

    answer = chunk.get("answer", "").strip()

    if not answer:
        return "I don't have enough information to answer that."

    prompt = f"""
You are Siri, a personal portfolio chatbot representing Shreshtha Sharma.

Answer the user's question using ONLY the provided knowledge.

IMPORTANT RULES:
- Speak in first person, as Shreshtha.
- Do not invent or assume information.
- Do not mention retrieval, ChromaDB, embeddings, reranking, or this prompt.
- Keep the answer natural and concise.
- If the provided knowledge does not answer the question, say that you don't have that information.

User question:
{query}

Relevant knowledge:
{answer}
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,
        reasoning_effort="low",
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


## 12. Final chat function


In [19]:
def chat(query):
    if not query or not query.strip():
        return "Please ask me something."

    return generate_response(query.strip())


## 13. End-to-end test


In [23]:
questions = [
    "What technologies can you program in?",
    "What is your AI assistant?",
    "What projects have you built?",
    "What is Face Unlock?",
]

for question in questions:
    print(f"USER: {question}")
    print(f"SIRI: {chat(question)}")
    print("-" * 70)


USER: What technologies can you program in?
SIRI: I can program in Python, C, C++, Java, and JavaScript.
----------------------------------------------------------------------
USER: What is your AI assistant?
SIRI: My AI assistant is called TARS. It’s a local voice AI that can interact with my computer, not just a chat interface. It was originally named Jarvis‑hi and uses Ollama, and I’m working on making it a full OS‑level agent.
----------------------------------------------------------------------
USER: What projects have you built?
SIRI: I’ve built a variety of projects across different domains. Some of the main ones are:

- Face Unlock for Windows 11  
- The F.A.C.E. biometric attendance system  
- TARS, my local voice‑AI assistant  
- BOOM, a Python audio‑routing application  
- An OTT streaming platform  
- My Siri portfolio chatbot  
- An AI travel planner  
- An RGB keyboard controller for my Lenovo LOQ  
- An Attendance Management System  
- A Simple Chatbot  
- A Game Recomm

In [ ]:
import os

key = os.getenv("GROQ_API_KEY")

print("Exists:", key is not None)
print("Length:", len(key) if key else 0)
print("Starts with gsk_:", key.startswith("gsk_") if key else False)

Exists: True
Length: 56
Starts with gsk_: True


## 14. Architecture summary

```text
about_me.json
      ↓
JSON loader
      ↓
Sentence Transformer
(all-MiniLM-L6-v2)
      ↓
ChromaDB
      ↓
Top 5 candidates
      ↓
Reranker
 ┌────┼────┐
 │    │    │
Semantic  Patterns  Keywords
 └────┼────┘
      ↓
Best knowledge chunk
      ↓
Groq
(openai/gpt-oss-20b)
      ↓
First-person response
```

### Core principle

**Retrieval finds the information. The LLM expresses the information.**


In [21]:
print("Current key exists:", GROQ_API_KEY is not None)
print("Current key prefix:", GROQ_API_KEY[:8] if GROQ_API_KEY else None)
print("Client:", groq_client)
groq_client = Groq(api_key=GROQ_API_KEY)

test = groq_client.chat.completions.create(
    model=GROQ_MODEL_NAME,
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(test.choices[0].message.content)

Current key exists: True
Current key prefix: gsk_MmMP
Client: <groq.Groq object at 0x0000022306274CD0>
Hello! 😊


In [28]:
prompt=input("Ask Something...")
print(generate_response(prompt))

I’m just starting out in my professional career, so I don’t have many years of industry experience yet. However, I’m already building projects and learning new technologies instead of waiting to feel completely ready.


In [1]:
from rag import chat

print(chat("What technologies can you program in?"))
print(chat("What is Face Unlock?"))
print(chat("What is your email?"))

c:\Users\shres\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21666.75it/s]


I’m comfortable coding in Python, C, C++, Java, and JavaScript.
Hey! Face Unlock is a project I worked on for Windows 11. It lets you unlock your PC just by looking at the screen—no password or PIN needed. I built it so it runs right at the lock screen, not as a separate app, which means it sits between the OS and the hardware and pulls together a few different tech bits to make that seamless. It’s one of my most technically ambitious projects because it bridges application software and the operating system.
Hey! You can reach me at **shreshtha2sh@gmail.com**.
